# Visulization Examples
## Using Data from the API
The [`cpra.mp.data` module](github.com/pscedu/cpra.mp.data) provides visualization classes that load data from the Data and Analysis API. These classes generate an [Altair](https://altair-viz.github.io/) chart that requests data as CSVs from the API and manipulates them in the browser.

### Single Dataset
This example loads a single CSV containing daily salinity data and draws a [time series chart](https://github.com/pscedu/cpra.mp.data/blob/main/cpra/mp/data/visualizations/time_series_line.py).

In [1]:
from cpra.mp.data.visualizations.time_series_line import TimeSeriesLine

TimeSeriesLine(
    variable="sal",
    grid="hydro_compartment_v001",
    time_unit="daily",
    aggregate_type="mean",
    model_group_id=500,
    scenario_id=7,
    geography=[1560, 1546],
    width=1000,
    height=600,
).chart()

alt.VConcatChart(...)

### Multiple Datasets
This example loads multiple CSVs containing different datasets from the API. It then joins them together in the browser and draws a [sediment dynamics chart](https://github.com/pscedu/cpra.mp.data/blob/main/cpra/mp/data/visualizations/sediment_dynamics.py).

In [2]:
from cpra.mp.data.visualizations.sediment_dynamics import SedimentDynamics

SedimentDynamics(
    model_group_id=500,
    scenario_id=7,
    geography=[-90.6582329, 29.4417686],
    width=1000,
    height=600,
).chart()

alt.LayerChart(...)

## Using Data from the Filesystem
Currently the Data and Analysis API only operates on a single dataset per request. Some simple analysis operations (e.g., joining two datasets) can be perfomed in the browser, as demonstrated in the previous example. However, for more complex analysis operations or those that require reading large amounts of data, the data can be read directly from the filesystem using the `read_data` function.

In [3]:
from cpra.mp.data import read_data

elevation_rasters = read_data(
    variable="elev",
    grid="morph_pixel_v001",
    time_unit="annual",
    model_group_id=500,
    scenario_id=7,
    calendar_year=(2020, 2050, 2070)
).collect()
elevation_rasters

variable,grid,time_unit,model_group_id,scenario_id,calendar_year,path
str,str,str,i32,i32,i32,str
"""elev""","""morph_pixel_v001""","""annual""",500,7,2020,"""/ocean/projects/bcs200002p/sha…"
"""elev""","""morph_pixel_v001""","""annual""",500,7,2050,"""/ocean/projects/bcs200002p/sha…"
"""elev""","""morph_pixel_v001""","""annual""",500,7,2070,"""/ocean/projects/bcs200002p/sha…"


In [4]:
import rasterio as rio
import polars as pl
from cpra.mp.data import vrt

HC_INDEX_RASTER = "zip+file:///ocean/projects/bcs200002p/ewhite12/MP2023/ICM/S07/G500/geomorph/input/MP2023_S00_G000_C000_U00_V00_SLA_I_00_00_W_comp30.tif.zip"

agg_expressions = {str(year): pl.col(str(year)).min() for year in [2020, 2050, 2070]}
window_mins_df = None

with rio.open(HC_INDEX_RASTER) as index_src, rio.open(vrt(elevation_rasters["path"])) as elevation_src:
    for block_index, window in elevation_src.block_windows(1):
        window_df = (
            pl.DataFrame({
                "fid": index_src.read(1, window=window).flatten(),
                "2020": elevation_src.read(1, window=window).flatten(),
                "2050": elevation_src.read(2, window=window).flatten(),
                "2070": elevation_src.read(3, window=window).flatten(),
            })
            .filter(pl.col("fid") > -1)
            .group_by("fid")
            .agg(**agg_expressions)
        )
        if window_mins_df is None:
            window_mins_df = window_df
        else:
            window_mins_df = pl.concat([window_mins_df, window_df])
        
mins_df = (
    window_mins_df
        .group_by("fid")
        .agg(**agg_expressions)
)
mins_df

fid,2020,2050,2070
i16,f32,f32,f32
363,-1.516043,-0.803471,-0.766199
39,-1.145567,-1.178721,-1.199498
715,-9.763021,-9.996318,-10.150065
707,-14.3045,-14.3045,-14.3045
584,-11.606469,-11.676934,-11.72482
…,…,…,…
1220,-2.17688,-2.206532,-2.226805
1117,-1.261503,-1.31864,-1.356229
1559,-4.389638,-4.389638,-4.389638


In [5]:
import altair as alt
import json

GRID = "/ocean/projects/bcs200002p/shared/grids/hydro_compartment_v001.json"

with open(GRID, "r") as grid:
    topojson = json.load(grid)

value_columns = ["2020", "2050", "2070"]
chart = (
    alt.Chart(alt.Data(values=topojson, format=alt.DataFormat(feature="hydro_compartment_v001", type="topojson")))
        .mark_geoshape()
        .encode(
            alt.Color(
                alt.repeat("row"),
                type="quantitative",
                scale=alt.Scale(domainMin=-5, scheme="browns")
            ),
            alt.Tooltip(alt.repeat("row")),
        )
        .transform_lookup(lookup="id", from_=alt.LookupData(mins_df, "fid", value_columns))
        .properties(width=1000, height=400)
        .project(type="identity", reflectY=True)
        .repeat(row=value_columns)
)
chart

alt.RepeatChart(...)